In [57]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Load PDF
loader = PyPDFLoader("short_story.pdf")
docs = loader.load()

# Split text
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

# Embeddings (local)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Store in FAISS
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [58]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    temperature=0.2
)

In [72]:
from langchain_community.vectorstores import FAISS
import os

if not os.path.exists("faiss_index"):
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local("faiss_index")
else:
    vectorstore = FAISS.load_local(
        "faiss_index",
        embeddings,
        allow_dangerous_deserialization=True
    )
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


def retrieve_docs(query):
    if isinstance(query, dict):
        query = query.get("query") or query.get("action_input") or str(query)
    
    print(f"--- Document Search Tool Triggered with Query: {query} ---")
    
    try:
        found_docs = retriever.invoke(query)
        
        if not found_docs:
            return "No relevant information found in the document."
            
        context = "\n\n".join([doc.page_content for doc in found_docs])
        return context
        
    except Exception as e:
        return f"Error during retrieval: {str(e)}"

In [60]:
from docx import Document
import os
from datetime import datetime

def create_word_doc(content, filename="generated.docx", folder="generated_docs"):
    os.makedirs(folder, exist_ok=True)

    filepath = os.path.join(folder, filename)

    doc = Document()

    # Title
    doc.add_heading("Generated Document", 0)

    # Content (split into paragraphs)
    for line in content.split("\n"):
        doc.add_paragraph(line)

    doc.save(filepath)

    return f"Document saved at: {filepath}"

In [61]:
from langchain_core.tools import Tool

def word_tool(query):
    prompt = f"""
Create a well-structured document:

{query}

    Format:
    - Title
    - Sections
    - Bullet points if needed
    """

    content = llm.invoke(prompt).content  # FIXED

    filename = f"doc_{datetime.now().strftime('%Y%m%d_%H%M%S')}.docx"
    return create_word_doc(content, filename)

word_tool_instance = Tool(
    name="Word Document Generator",
    func=word_tool,
    description="Creates a Word document from a user request"
)

In [62]:
import math

def tool_calculator(query):     # This tool allows the agent to solve arithmetic expressions
    try:
        result = eval(query, {"__builtins__": None}, {"sqrt": math.sqrt})
        return str(result)
    except:
        return "Invalid arithmetic expression"

calculator_tool = Tool(
    name="Calculator",
    func=tool_calculator,
    description="Solve arithmetic problems like addition, subtraction, multiplication, and division"
)

In [63]:
import wikipedia

def tool_wikipedia(query):
    try:
        return wikipedia.summary(query, sentences=5)
    except Exception as e:
        return f"Error: {str(e)}"

wikipedia_tool = Tool(
    name="Wikipedia Extractor",
    func=tool_wikipedia,
    description="Gets Wikipedia summaries"
)

In [64]:
def wiki_doc_creator(query):          # This tool extracts Wikipedia info and creates a Word document from it
    try:
        summary = wikipedia.summary(query, sentences=10)
    except:
        summary = "Wikipedia article not found."

    prompt = f"""
    Create a structured document using this information:

    {summary}

    Format:
    - Title
    - Sections
    - Bullet points if needed
    """

    content = llm.invoke(prompt).content

    filename = f"wiki_doc_{datetime.now().strftime('%Y%m%d_%H%M%S')}.docx"

    return create_word_doc(content, filename)

wikipedia_doc_tool = Tool(
    name="Wikipedia Document Creator",
    func=wiki_doc_creator,
    description="Creates a Word document using information extracted from Wikipedia"
)

In [73]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

doc_retrieval_tool = Tool(
    name="Document Search",
    func=retrieve_docs,
    description="Search PDF documents"
)

tools = [
    word_tool_instance,
    doc_retrieval_tool,
    calculator_tool,
    wikipedia_tool,
    wikipedia_doc_tool
]

system_message = """
You are an intelligent AI assistant with access to tools.

AVAILABLE TOOLS:
- Document Search → for answering questions from uploaded PDFs or documents
- Word Document Generator → for creating .docx files
- Calculator → solve arithmetic problems
- Wikipedia Extractor → extract information from Wikipedia
- Wikipedia Document Creator → create Word documents using Wikipedia data

DECISION RULES:
1. If the question is about documents or requires context → use Document Search
2. If the user asks to create/generate/write a file → use Word Document Generator
3. If the question is general knowledge → answer directly
4. If the user asks for calculations → use Calculator
5. If the user asks for Wikipedia information → use Wikipedia Extractor
6. If the user asks to create a document from Wikipedia → use Wikipedia Document Creator
7. Always provide a final answer after using tools, make your answer clean and clear,and summarize information when needed.
8. If you use a tool, explain your reasoning in the final answer.
"""

agent = create_react_agent(
    llm,
    tools,
    checkpointer=memory
)

In [66]:
def run_agent(query):
    config = {"configurable": {"thread_id": "user-1"}}
    
    # We pass the system message as a specific 'system' message type
    response = agent.invoke(
        {"messages": [("system", system_message), ("user", query)]},
        config=config
    )
    
    return response["messages"][-1].content

In [67]:
query = "What is 12 + (6^2 / 4) * 3 - 5 ?"
print(run_agent(query))



The expression $12 + (6^2 / 4) \times 3 - 5$ simplifies as follows:
1. $6^2 = 36$
2. $36 / 4 = 9$
3. $9 \times 3 = 27$
4. $12 + 27 = 39$
5. $39 - 5 = 34$

**Final Answer**: 34


In [68]:
query = "Can you create a document about the history of the internet using Wikipedia?"
print(run_agent(query))



The document titled "History of the Internet" has been successfully created using Wikipedia data and saved at:  
**generated_docs\wiki_doc_20260423_235548.docx**

You can download this Word document to review the comprehensive historical overview of the internet's development. Let me know if you'd like further assistance!


In [69]:
query = "What is the internet?"
print(run_agent(query))



The internet is a global network of interconnected computers that communicate using standardized protocols (such as TCP/IP) to share data, resources, and information. It was developed in the late 1960s as ARPANET by the U.S. Department of Defense and has since evolved into a worldwide system connecting billions of devices for communication, commerce, education, entertainment, and more. Today, it serves as the foundation for modern digital life, including social media, cloud computing, and real-time interactions across the globe.


In [70]:
query = "Can you explain the concept of artificial intelligence and give some examples?"
print(run_agent(query))



Artificial intelligence (AI) refers to the capability of machines to perform tasks that typically require human intelligence, such as learning, problem-solving, pattern recognition, and decision-making. AI systems are designed to mimic cognitive functions by analyzing data, adapting to new inputs, and executing complex operations.

**Examples of AI in action**:
1. **Chatbots and Virtual Assistants**: Tools like Siri, Alexa, and ChatGPT use natural language processing to understand and respond to user queries.
2. **Recommendation Systems**: Platforms like Netflix and Amazon leverage AI to suggest movies, shows, or products based on user preferences and behavior.
3. **Image and Speech Recognition**: Apps like Google Photos use AI to identify objects, people, and scenes in images, while voice assistants like Google Assistant process spoken commands.
4. **Autonomous Vehicles**: Self-driving cars (e.g., Tesla’s Autopilot) rely on AI to interpret sensor data, navigate roads, and make real-

In [75]:
query = "Who is the protagonist in the uploaded short story and what are their main motivations?"
print(run_agent(query))

--- Document Search Tool Triggered with Query: Who is the protagonist in the short story and what are their main motivations? ---


The protagonist in the uploaded short story is **Eli**, a young radio enthusiast. His main motivation is **curiosity**, driven by the mysterious signal he receives from an old radio tower. The story reveals that Eli follows the coordinates hidden in static, discovers a cabin with a photo of a man who looks exactly like him, and is compelled to investigate the signal’s origin—a mystery tied to the past. This curiosity leads him to uncover the signal’s final transmission, which goes silent after that night.

**Final Answer**:  
The protagonist is **Eli**, and his main motivation is **curiosity** about the mysterious signal and its connection to the past.
